# El Experimento del Doble Puente y el Modelo ACO 🐜

Basado en la sección 1.1 del libro de Dorigo, este cuaderno explica las ecuaciones diferenciales y estocásticas que modelan cómo las hormigas encuentran el camino más corto.

### 1. La Ecuación de Decisión (Ecuación 1.1 de la imagen)

Cuando una hormiga llega a la bifurcación, usa esta fórmula probabilística para decidir si toma la rama corta ($s$) o la larga ($l$):

$$p_{is}(t) = \frac{(t_s + \varphi_{is}(t))^\alpha}{(t_s + \varphi_{is}(t))^\alpha + (t_s + \varphi_{il}(t))^\alpha}$$

**¿Qué significa cada variable?**
* **$p_{is}(t)$**: Es la probabilidad de elegir la rama corta ($s$) en el tiempo $t$.
* **$\varphi_{is}(t)$ y $\varphi_{il}(t)$**: Es la cantidad de feromonas acumuladas en la rama corta y larga, respectivamente. El texto dice que es proporcional al número de hormigas que han pasado.
* **$t_s$**: Es una constante de atracción inicial. Evita que la ecuación se rompa dividiendo por cero cuando no hay feromonas al principio.
* **$\alpha = 2$**: ¡Este es el factor crítico! Al elevar las feromonas al cuadrado, el modelo hace que la decisión sea **no lineal**. Si un camino tiene un poco más de feromonas que el otro, el exponente 2 magnifica esa ventaja enormemente.

### 2. El Secreto del Tiempo (Ecuaciones 1.2 y 1.3)

Si las hormigas eligen de forma casi aleatoria al principio, ¿por qué siempre gana el camino corto? La respuesta está en las ecuaciones de evolución temporal:

$$d\varphi_{is}/dt = \psi p_{js}(t - t_s) + \psi p_{is}(t)$$
$$d\varphi_{il}/dt = \psi p_{jl}(t - r \cdot t_s) + \psi p_{il}(t)$$

El secreto es el **retraso (delay)** en los paréntesis:
* **$t - t_s$**: Las hormigas que toman la rama corta tardan un tiempo $t_s$ en cruzar y volver, depositando feromonas rápidamente.
* **$t - r \cdot t_s$**: Las hormigas que toman la rama larga tardan $r$ veces más (donde $r > 1$). 

Como las hormigas del camino corto regresan antes, depositan feromonas más rápido, inclinando la Ecuación 1.1 a su favor antes de que las hormigas del camino largo puedan siquiera regresar.

---
## 💻 Simulación Interactiva del Sistema Estocástico

Vamos a programar exactamente este modelo. 
En esta simulación lanzaremos varias hormigas por segundo ($\psi$). Podrás ajustar qué tan largo es el camino largo ($r$) y el parámetro de no linealidad ($\alpha$).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

In [2]:
def simular_doble_puente(ratio_longitud=2.0, alpha=2.0, hormigas_por_seg=2, tiempo_total=150):
    t_s = 20.0  # Constante base del modelo de la imagen
    
    # Historial de feromonas (empiezan en 0)
    phi_s = [0.0]
    phi_l = [0.0]
    
    # Historial de probabilidades
    prob_s_hist = []
    
    # Sistema de retraso (simula el t - t_s de las ecs 1.2 y 1.3)
    # Guardamos eventos de depósito futuro: [tiempo_deposito, rama]
    eventos_feromonas = []
    
    tiempo_cruce_corto = 5  # Representa t_s en la simulación
    tiempo_cruce_largo = int(tiempo_cruce_corto * ratio_longitud)  # Representa r * t_s
    
    for t in range(tiempo_total):
        # 1. Calcular probabilidad con la Ecuación 1.1 de la imagen
        numerador_s = (t_s + phi_s[-1]) ** alpha
        numerador_l = (t_s + phi_l[-1]) ** alpha
        p_s = numerador_s / (numerador_s + numerador_l)
        p_l = 1.0 - p_s
        
        prob_s_hist.append(p_s)
        
        # 2. Las hormigas toman la decisión estocástica
        for _ in range(hormigas_por_seg):
            eleccion = np.random.choice(['S', 'L'], p=[p_s, p_l])
            
            if eleccion == 'S':
                # La hormiga deja rastro al llegar al otro lado (pasado el tiempo de cruce)
                eventos_feromonas.append((t + tiempo_cruce_corto, 'S'))
            else:
                eventos_feromonas.append((t + tiempo_cruce_largo, 'L'))
                
        # 3. Actualizar feromonas con las hormigas que van llegando (Eq 1.2 y 1.3)
        nuevo_phi_s = phi_s[-1]
        nuevo_phi_l = phi_l[-1]
        
        for evento in eventos_feromonas.copy():
            if evento[0] == t:  # Si es el momento de depositar
                if evento[1] == 'S':
                    nuevo_phi_s += 1
                else:
                    nuevo_phi_l += 1
                eventos_feromonas.remove(evento)
                
        phi_s.append(nuevo_phi_s)
        phi_l.append(nuevo_phi_l)
        
    # Graficar resultados
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))
    
    # Gráfica de Probabilidad de la rama corta
    ax1.plot(prob_s_hist, color='green', linewidth=2)
    ax1.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
    ax1.set_title(rf'Probabilidad de elegir el camino corto ($p_{{is}}$) | $\alpha$={alpha}')
    ax1.set_ylabel('Probabilidad')
    ax1.set_ylim(0, 1.05)
    ax1.grid(True, alpha=0.3)
    
    # Gráfica de acumulación de feromonas (Sin evaporación como dice el texto)
    ax2.plot(phi_s[1:], label=r'Feromonas Rama Corta ($\varphi_s$)', color='blue')
    ax2.plot(phi_l[1:], label=r'Feromonas Rama Larga ($\varphi_l$)', color='red')
    ax2.set_title('Acumulación de Feromonas en el tiempo')
    ax2.set_xlabel('Tiempo ($t$)')
    ax2.set_ylabel('Cantidad de feromonas')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Interfaz gráfica con sliders
interact(simular_doble_puente, 
         ratio_longitud=FloatSlider(min=1.1, max=5.0, step=0.1, value=2.0, description='Ratio Longitud (r):'),
         alpha=FloatSlider(min=0.5, max=4.0, step=0.5, value=2.0, description=r'No-linealidad ($\alpha$):'));

interactive(children=(FloatSlider(value=2.0, description='Ratio Longitud (r):', max=5.0, min=1.1), FloatSlider…

### 3. ¿Cómo ejecutar esta simulación y qué esperar?

**Pasos para ejecutar:**
1. Ejecuta todas las celdas de este cuaderno en orden.
2. Al llegar a la última celda, aparecerá la gráfica acompañada de dos barras deslizadoras (sliders).

**Qué esperar al jugar con los controles:**
* **La línea verde de probabilidad:** Representa qué tan probable es que la colonia elija el camino corto. Al inicio la verás oscilar cerca del 0.5 (decisión estocástica dividida), pero conforme pase el tiempo, se disparará hacia 1.0. ¡La colonia ha convergido al camino óptimo!
* **Las líneas de feromonas (azul y roja):** La línea azul siempre tomará la delantera porque sus hormigas completan el viaje primero. Esa pequeña ventaja temprana es la que alimenta la Ecuación 1.1 a su favor.
* **Cambia el valor de $\alpha$:** Si deslizas $\alpha$ a un valor de `1.0`, verás que la colonia duda mucho más y le toma más tiempo alcanzar la convergencia. Si usas un $\alpha$ alto (ej. `3.0`), la colonia es extremadamente sensible a las feromonas y convergirá agresivamente.
* **Cambia el Ratio Longitud ($r$):** Si haces el camino largo mucho más largo (ej. $r = 4.0$), el retraso se vuelve tan grande que la rama corta ganará de manera casi instantánea.

### 4. Resumen del Algoritmo (Pseudocódigo)

Aquí tienes la lógica fundamental de este modelo estocástico reducida a pseudocódigo, lo que permite apreciar la simpleza de las reglas locales que producen una inteligencia global compleja:

```text
INICIALIZAR feromona_corta = 0
INICIALIZAR feromona_larga = 0
DEFINIR tiempo_cruce_corto = 5  // Unidades de tiempo (t_s)
DEFINIR tiempo_cruce_largo = 10 // Representa r * t_s

MIENTRAS la simulación esté en ejecución (avanzando en el tiempo t):
    
    // 1. Calcular atractivo relativo basado en las feromonas actuales
    atraccion_corta = (constante + feromona_corta) ^ alpha
    atraccion_larga = (constante + feromona_larga) ^ alpha
    
    prob_corta = atraccion_corta / (atraccion_corta + atraccion_larga)
    
    // 2. Proceso de decisión estocástico para las nuevas hormigas
    PARA CADA hormiga que llega a la bifurcación en el tiempo t:
        generar_numero_aleatorio entre 0 y 1
        SI numero_aleatorio <= prob_corta:
            // Tomó el camino corto
            PROGRAMAR_EVENTO: sumar feromona_corta en (t + tiempo_cruce_corto)
        SINO:
            // Tomó el camino largo
            PROGRAMAR_EVENTO: sumar feromona_larga en (t + tiempo_cruce_largo)
            
    // 3. Sistema de retraso (Delay System)
    REVISAR EVENTOS PROGRAMADOS:
    SI hay feromonas programadas para ser depositadas en el tiempo t:
        Actualizar feromona_corta o feromona_larga respectivamente
```